# Lab: Delta Tables

**Course 1, Week 3: Delta Lake & Workflows**

## Objectives
- Create and manage Delta tables
- Perform INSERT, UPDATE, and MERGE operations
- Use time travel to query historical data
- Understand schema enforcement and evolution


## What Is Delta Lake?

Delta Lake is an open-source storage framework that brings reliability to data lakes:

- **ACID Transactions:** Every write is atomic — no partial/corrupt data
- **Schema Enforcement:** Prevents bad data from entering tables
- **Schema Evolution:** Safely add new columns over time
- **Time Travel:** Query previous versions of your data
- **Unified Batch + Streaming:** Same table for both workloads
- **Audit History:** Full log of every operation

## Delta Lake Architecture

```
Delta Table
├── _delta_log/              # Transaction log (JSON + Parquet)
│   ├── 00000000000000.json  # Version 0
│   ├── 00000000000001.json  # Version 1
│   └── 00000000000010.checkpoint.parquet  # Checkpoint
└── part-00000-*.parquet     # Data files (standard Parquet)
```

The **transaction log** (`_delta_log/`) is what gives Delta Lake its superpowers.
Every change is recorded as a JSON commit file.


## Key Concepts for Certification

| Concept | What It Does | Why It Matters |
|---------|-------------|----------------|
| ACID Transactions | Atomic, consistent writes | No corrupt data |
| Schema Enforcement | Validates data on write | Data quality |
| Schema Evolution | Add columns safely | Agile development |
| Time Travel | Query historical versions | Auditing, rollback |
| MERGE | Upsert in one operation | Efficient CDC |
| Transaction Log | Records every change | Audit trail |

## Part 1: Create a Delta Table

EXERCISE: Create a Delta table for an inventory system.

In [0]:
# EXERCISE: Create an inventory DataFrame with columns:
# sku (string), product_name (string), category (string), quantity (int), unit_price (double)
# Include at least 6 products across 2+ categories
# YOUR CODE HERE

inventory_data = [
    ("SKU001", "Laptop", "Electronics", 15, 899.99),
    ("SKU002", "Mouse", "Electronics", 50, 25.50),
    ("SKU003", "Desk Chair", "Furniture", 10, 150.00),
    ("SKU004", "Notebook", "Office Supplies", 100, 3.99),
    ("SKU005", "Pen Set", "Office Supplies", 75, 12.50),
    ("SKU006", "Monitor", "Electronics", 20, 299.99)
]

# Pass the schema as a simple DDL string string
schema_str = "sku STRING, product_name STRING, category STRING, quantity INT, unit_price DOUBLE"

inventory_df = spark.createDataFrame(inventory_data, schema=schema_str)

inventory_df.printSchema()

root
 |-- sku: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)



In [0]:
# EXERCISE: Write the DataFrame as a Delta table named "lab_inventory"
# YOUR CODE HERE

inventory_df.write.format("delta").saveAsTable("lab_inventory")

In [0]:
%sql
-- EXERCISE: Verify the table was created
-- YOUR CODE HERE

SELECT * FROM lab_inventory

sku,product_name,category,quantity,unit_price
SKU001,Laptop,Electronics,15,899.99
SKU002,Mouse,Electronics,50,25.5
SKU003,Desk Chair,Furniture,10,150.0
SKU004,Notebook,Office Supplies,100,3.99
SKU005,Pen Set,Office Supplies,75,12.5
SKU006,Monitor,Electronics,20,299.99


## Part 2: INSERT New Records

EXERCISE: Add new products to the inventory.

In [0]:
# EXERCISE: Create a DataFrame with 2 new products and INSERT (append) them
# YOUR CODE HERE

new_data = [
    ("SKU007", "Keyboard", "Electronics", 30, 49.99),
    ("SKU008", "Stapler", "Office Supplies", 40, 7.50)
]

schema_str = "sku string, product_name string, category string, quantity int, unit_price double"
new_df = spark.createDataFrame(new_data, schema=schema_str)

new_df.write.format("delta").mode("append").saveAsTable("lab_inventory")


In [0]:
%sql
-- Verify the insert

SELECT *  FROM lab_inventory

sku,product_name,category,quantity,unit_price
SKU001,Laptop,Electronics,15,899.99
SKU002,Mouse,Electronics,50,25.5
SKU003,Desk Chair,Furniture,10,150.0
SKU004,Notebook,Office Supplies,100,3.99
SKU005,Pen Set,Office Supplies,75,12.5
SKU006,Monitor,Electronics,20,299.99
SKU007,Keyboard,Electronics,30,49.99
SKU008,Stapler,Office Supplies,40,7.5


## Part 3: UPDATE Records

EXERCISE: Update prices for a category.


In [0]:

%sql
-- EXERCISE: Increase all prices by 5% for one category of your choice
-- YOUR CODE HERE 

UPDATE lab_inventory 
SET unit_price = unit_price * 1.05 
WHERE category = 'Electronics'

num_affected_rows
4


In [0]:
%sql
-- Verify the update

SELECT *  FROM lab_inventory

sku,product_name,category,quantity,unit_price
SKU001,Laptop,Electronics,15,944.9895
SKU002,Mouse,Electronics,50,26.775000000000002
SKU006,Monitor,Electronics,20,314.9895
SKU007,Keyboard,Electronics,30,52.48950000000001
SKU003,Desk Chair,Furniture,10,150.0
SKU004,Notebook,Office Supplies,100,3.99
SKU005,Pen Set,Office Supplies,75,12.5
SKU008,Stapler,Office Supplies,40,7.5


## Part 4: MERGE (Upsert)

EXERCISE: Perform a MERGE operation to update existing and insert new products.

In [0]:
%sql
-- EXERCISE: Create a source view with updates and new records
-- Include at least 1 existing SKU (update) and 1 new SKU (insert)
-- YOUR CODE HERE

CREATE OR REPLACE TEMP VIEW source_inventory AS
SELECT * FROM VALUES
  ('SKU001', 'Gaming Laptop', 'Electronics', 10, 999.99), -- existing SKU (update)
  ('SKU009', 'Wireless Mouse', 'Electronics', 25, 35.00)   -- new SKU (insert)
AS tab(sku, product_name, category, quantity, unit_price);

In [0]:
%sql
-- EXERCISE: Write a MERGE statement
-- Match on SKU
-- WHEN MATCHED: update quantity and unit_price
-- WHEN NOT MATCHED: insert all columns
-- YOUR CODE HERE


MERGE INTO lab_inventory AS target
USING source_inventory AS source
ON target.sku = source.sku
WHEN MATCHED THEN
  UPDATE SET 
    target.quantity = source.quantity, 
    target.unit_price = source.unit_price
WHEN NOT MATCHED THEN
  INSERT (sku, product_name, category, quantity, unit_price)
  VALUES (source.sku, source.product_name, source.category, source.quantity, source.unit_price);

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2,1,0,1


## Part 5: Time Travel

EXERCISE: Query previous versions of the table.


In [0]:
%sql
-- EXERCISE: View the full history of the table
-- YOUR CODE HERE

DESCRIBE HISTORY lab_inventory

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
5,2026-09-02T03:49:42.000Z,78511136629920,delacruzdaniellemarie@yahoo.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(615503590206839),36bca802-ad5e-49fd-b3e2-2352f4647dba,0902-013027-fgio6uhb-v2n,4,SnapshotIsolation,false,"Map(numRemovedFiles -> 3, numRemovedBytes -> 5525, p25FileSize -> 2057, numDeletionVectorsRemoved -> 1, minFileSize -> 2057, numAddedFiles -> 1, maxFileSize -> 2057, p75FileSize -> 2057, p50FileSize -> 2057, numAddedBytes -> 2057)",null,Databricks-Runtime/19.5.x-aarch64-photon-scala2.13
4,2026-09-02T03:49:41.000Z,78511136629920,delacruzdaniellemarie@yahoo.com,MERGE,"Map(predicate -> [""(sku#13851 = sku#13836)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(615503590206839),36bca802-ad5e-49fd-b3e2-2352f4647dba,0902-013027-fgio6uhb-v2n,3,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 2, numTargetBytesAdded -> 3510, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 1, executionTimeMs -> 4421, materializeSourceTimeMs -> 390, numTargetRowsInserted -> 1, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1681, numTargetRowsUpdated -> 1, numOutputRows -> 2, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2286)",null,Databricks-Runtime/19.5.x-aarch64-photon-scala2.13
3,2026-09-02T03:43:28.000Z,78511136629920,delacruzdaniellemarie@yahoo.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(615503590206839),c8509dab-d2fe-4a5b-ab54-52c44509eac0,0902-013027-fgio6uhb-v2n,2,SnapshotIsolation,false,"Map(numRemovedFiles -> 3, numRemovedBytes -> 5674, p25FileSize -> 2015, numDeletionVectorsRemoved -> 2, minFileSize -> 2015, numAddedFiles -> 1, maxFileSize -> 2015, p75FileSize -> 2015, p50FileSize -> 2015, numAddedBytes -> 2015)",null,Databricks-Runtime/19.5.x-aarch64-photon-scala2.13
2,2026-09-02T03:43:25.000Z,78511136629920,delacruzdaniellemarie@yahoo.com,UPDATE,"Map(predicate -> [""(category#13138 = Electronics)""])",null,List(615503590206839),c8509dab-d2fe-4a5b-ab54-52c44509eac0,0902-013027-fgio6uhb-v2n,1,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 2, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 4118, numDeletionVectorsUpdated -> 0, scanTimeMs -> 2058, numAddedFiles -> 1, numUpdatedRows -> 4, numAddedBytes -> 1883, rewriteTimeMs -> 2031)",null,Databricks-Runtime/19.5.x-aarch64-photon-scala2.13
1,2026-09-02T03:41:05.000Z,78511136629920,delacruzdaniellemarie@yahoo.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(615503590206839),51d27e6c-fc42-44de-bcce-ce10f7114462,0902-013027-fgio6uhb-v2n,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 2, numOutputBytes -> 1823)",null,Databricks-Runtime/19.5.x-aarch64-photon-scala2.13
0,2026-09-02T03:30:41.000Z,78511136629920,delacruzdaniellemarie@yahoo.com,CREATE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(615503590206839),3e47258e-5a7b-42f6-a779-7127f1f57f87,0902-013027-fgio6uhb-v2n,null,WriteSeria

In [0]:
%sql
-- EXERCISE: Query the original version (version 0) and compare with current
-- YOUR CODE HERE

SELECT * FROM lab_inventory VERSION AS OF 0

sku,product_name,category,quantity,unit_price
SKU001,Laptop,Electronics,15,899.99
SKU002,Mouse,Electronics,50,25.5
SKU003,Desk Chair,Furniture,10,150.0
SKU004,Notebook,Office Supplies,100,3.99
SKU005,Pen Set,Office Supplies,75,12.5
SKU006,Monitor,Electronics,20,299.99


In [0]:
%sql
-- EXERCISE: Write a query that shows price changes between version 0 and current
-- Join current with VERSION AS OF 0 on SKU
-- Show: sku, product_name, original_price, current_price, price_change
-- YOUR CODE HERE

SELECT 
    curr.sku,
    curr.product_name,
    v0.unit_price AS orig_unit_price,
    curr.unit_price AS curr_unit_price,
    curr.unit_price - v0.unit_price AS price_change
FROM lab_inventory AS curr
JOIN lab_inventory VERSION AS OF 0 as v0
ON curr.sku = v0.sku


sku,product_name,orig_unit_price,curr_unit_price,price_change
SKU001,Laptop,899.99,999.99,100.0
SKU002,Mouse,25.5,26.775000000000002,1.2750000000000021
SKU003,Desk Chair,150.0,150.0,0.0
SKU004,Notebook,3.99,3.99,0.0
SKU005,Pen Set,12.5,12.5,0.0
SKU006,Monitor,299.99,314.9895,14.999500000000012


## Part 6: Schema Enforcement

EXERCISE: Test that Delta Lake enforces the schema.

In [0]:
# EXERCISE: Try to insert a DataFrame with wrong schema (missing columns)
# This should FAIL - verify that Delta Lake catches the error
# YOUR CODE HERE
# Hint: Try creating a DataFrame with only 2-3 columns and appending to lab_inventory

# Missing 'category', extra unauthorized column, and wrong data type for quantity
bad_data = [("SKU988", "Broken Item", "NOT_AN_INT", 99.99, "EXTRA_COL")]
bad_schema = "sku string, product_name string, quantity string, unit_price double, bad_col string"

bad_df = spark.createDataFrame(bad_data, schema=bad_schema)

# This will hard fail with a schema enforcement / analysis error
bad_df.write.format("delta").mode("append").saveAsTable("lab_inventory")

---------------------------------------------------------------------------
NumberFormatException                     Traceback (most recent call last)
File <command-7125328424630308>, line 13
     10 bad_df = spark.createDataFrame(bad_data, schema=bad_schema)
     12 # This will hard fail with a schema enforcement / analysis error
---> 13 bad_df.write.format("delta").mode("append").saveAsTable("lab_inventory")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:737, in DataFrameWriter.saveAsTable(self, name, format, mode, partitionBy, **options)
    735 self._write.table_name = name
    736 self._write.table_save_method = "save_as_table"
--> 737 _, _, ei = self._spark.client.execute_command(
    738     self._write.command(self._spark.client), self._write.observations
    739 )
    740 self._callback(ei)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:1538, in SparkConnectClient.execute_command(self, command,

In [0]:
%sql

SELECT * FROM lab_inventory

sku,product_name,category,quantity,unit_price
SKU002,Mouse,Electronics,50,26.775000000000002
SKU006,Monitor,Electronics,20,314.9895
SKU007,Keyboard,Electronics,30,52.48950000000001
SKU003,Desk Chair,Furniture,10,150.0
SKU004,Notebook,Office Supplies,100,3.99
SKU005,Pen Set,Office Supplies,75,12.5
SKU008,Stapler,Office Supplies,40,7.5
SKU001,Laptop,Electronics,10,999.99
SKU009,Wireless Mouse,Electronics,25,35.0


## Validation

In [0]:
def validate_lab():
    """Validate lab completion."""
    checks = []

    # Check 1: Table exists
    try:
        df = spark.sql("SELECT * FROM lab_inventory")
        checks.append(("Delta table exists", True))
    except Exception:
        checks.append(("Delta table exists", False))
        df = None

    # Check 2: Table has data
    if df:
        checks.append(("Table has data", df.count() >= 6))

    # Check 3: Multiple versions exist (operations were performed)
    try:
        history = spark.sql("DESCRIBE HISTORY lab_inventory")
        checks.append(("Multiple versions (DML performed)", history.count() >= 3))
    except Exception:
        checks.append(("Multiple versions (DML performed)", False))

    print("Lab Validation Results:")
    print("-" * 40)
    all_passed = True
    for name, passed in checks:
        status = "PASS" if passed else "FAIL"
        print(f"  [{status}] {name}")
        if not passed:
            all_passed = False

    if all_passed:
        print("\nAll checks passed! Lab complete.")
    else:
        print("\nSome checks failed. Review your code above.")

validate_lab()

Lab Validation Results:
----------------------------------------
  [PASS] Delta table exists
  [PASS] Table has data
  [PASS] Multiple versions (DML performed)

All checks passed! Lab complete.


In [0]:
# Clean up
try:
    spark.sql("DROP TABLE IF EXISTS lab_inventory")
except Exception:
    pass